# HECKTOR 2026 · nnU-Net 完整训练流程

**用法：从上到下，一格一格运行。** 每一格跑完、没有红色报错，再跑下一格。

- 开始前：Colab 菜单 **代码执行程序 → 更改运行时类型 → 选 GPU**（T4 就行，有 A100 更快）。
- 开始前：在 Google Drive 的「Shared with me」里，右键 `HECKTOR_2026` → **整理 → 添加快捷方式 → 我的云端硬盘**。
- **如果 Colab 断线了**：本地文件会被清空，但训练结果存在 Drive 上不会丢。重新从第 1 步跑到第 6 步，然后跑第 7b 步（接着上次继续训练）。
- **如果只是内核重启了**（比如第 6b 步内存爆了）：本地文件还在，只需要重跑第 1、2 步，再接着跑原来那一格。

## 第 1 步：检查 GPU，安装 nnU-Net

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q nnunetv2

from importlib.metadata import version
print("nnunetv2 版本:", version("nnunetv2"))

## 第 2 步：挂载 Drive，设置路径和环境变量

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# 原始数据（只读）
SRC = "/content/drive/MyDrive/HECKTOR_2026/HECKTOR 2026 Training Data"

# raw 和 preprocessed 放在 Colab 本地盘：读写快，但断线会清空
WORK = "/content/nnUNet_work"
# results 放在你自己的 Drive：训练好的模型永久保存，断线不丢
RESULTS = "/content/drive/MyDrive/nnunet_work/nnUNet_results"

os.environ["nnUNet_raw"] = f"{WORK}/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = f"{WORK}/nnUNet_preprocessed"
os.environ["nnUNet_results"] = RESULTS

# 训练时数据增强的进程数：Colab 只有约 12 GB 内存、2 个 CPU，开太多进程会把内存撑爆
os.environ["nnUNet_n_proc_DA"] = "2"

for k in ["nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"]:
    os.makedirs(os.environ[k], exist_ok=True)
    print(k, "=", os.environ[k])

assert os.path.isdir(SRC), "找不到原始数据！请先把 HECKTOR_2026 添加快捷方式到「我的云端硬盘」"
print("\n✅ 路径设置完成")

## 第 3 步：选病人

`N_PATIENTS` 是这次用多少个病人。**先用 50 个**：Colab 本地盘只有约 110 GB，全部 782 个病人的原始文件太大，再加预处理文件很容易塞满。跑通以后再加。

`MAX_SLICES`：层数超过 350 的大扫描（约 93 个）会撑爆免费 Colab 的内存，先排除。如果用内存更大的机器跑，可以把它调大。

In [ ]:
import random
import nibabel as nib
from collections import Counter

N_PATIENTS = 50
MAX_SLICES = 350   # 层数太多的大扫描会撑爆免费 Colab 的内存，先排除
DATASET_NAME = "Dataset501_HECKTOR"

all_patients, too_big = [], []
for p in sorted(os.listdir(SRC)):
    d = os.path.join(SRC, p)
    if p.startswith(".") or not os.path.isdir(d):
        continue
    need = [f"{d}/{p}__CT.nii.gz", f"{d}/{p}__PT.nii.gz", f"{d}/{p}.nii.gz"]
    if not all(os.path.exists(f) and os.path.getsize(f) > 0 for f in need):
        continue
    n_slices = nib.load(need[0]).shape[2]   # 只读文件头，很快
    if n_slices > MAX_SLICES:
        too_big.append(p)
    else:
        all_patients.append(p)

print("可用的病人:", len(all_patients))
print("因为太大被排除的病人:", len(too_big))

random.seed(0)
patients = sorted(random.sample(all_patients, min(N_PATIENTS, len(all_patients))))
print("这次使用:", len(patients), "个病人")
print("来自哪些医院:", dict(Counter(p.split("-")[0] for p in patients)))

## 第 4 步：复制成 nnU-Net 的格式（CT = _0000，PET = _0001）

In [ ]:
import shutil, time

imagesTr = f"{os.environ['nnUNet_raw']}/{DATASET_NAME}/imagesTr"
labelsTr = f"{os.environ['nnUNet_raw']}/{DATASET_NAME}/labelsTr"
os.makedirs(imagesTr, exist_ok=True)
os.makedirs(labelsTr, exist_ok=True)

# 删掉以前留下的、不在这次名单里的病人，避免数量对不上
keep = set(patients)
for folder in [imagesTr, labelsTr]:
    for f in os.listdir(folder):
        if f.split("_0000")[0].split("_0001")[0].replace(".nii.gz", "") not in keep:
            os.remove(os.path.join(folder, f))

def copy_ok(src, dst):
    # 已经完整复制过就跳过；复制完检查大小，防止出现 0 字节的坏文件
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        return
    shutil.copyfile(src, dst)
    if os.path.getsize(dst) != os.path.getsize(src):
        raise IOError(f"复制不完整: {dst}")

t0 = time.time()
for i, p in enumerate(patients, 1):
    d = os.path.join(SRC, p)
    copy_ok(f"{d}/{p}__CT.nii.gz", f"{imagesTr}/{p}_0000.nii.gz")
    copy_ok(f"{d}/{p}__PT.nii.gz", f"{imagesTr}/{p}_0001.nii.gz")
    copy_ok(f"{d}/{p}.nii.gz",     f"{labelsTr}/{p}.nii.gz")
    print(f"[{i}/{len(patients)}] {p}   已用 {(time.time() - t0) / 60:.1f} 分钟")

print("\nimagesTr 文件数:", len(os.listdir(imagesTr)), "（应该是病人数 × 2）")
print("labelsTr 文件数:", len(os.listdir(labelsTr)), "（应该等于病人数）")

## 第 5 步：写 dataset.json，抽查几个病人

In [ ]:
import json
import nibabel as nib
import numpy as np

dataset_json = {
    "channel_names": {"0": "CT", "1": "PET"},
    "labels": {"background": 0, "GTVp": 1, "GTVn": 2},
    "numTraining": len(patients),
    "file_ending": ".nii.gz",
}
with open(f"{os.environ['nnUNet_raw']}/{DATASET_NAME}/dataset.json", "w") as f:
    json.dump(dataset_json, f, indent=4)
print(json.dumps(dataset_json, indent=2))

print("\n抽查前 3 个病人（三个形状应该一样，标签值应该在 0/1/2 里）：")
for p in patients[:3]:
    ct = nib.load(f"{imagesTr}/{p}_0000.nii.gz")
    pt = nib.load(f"{imagesTr}/{p}_0001.nii.gz")
    lb = nib.load(f"{labelsTr}/{p}.nii.gz")
    print(p, "CT", ct.shape, "PET", pt.shape, "标签", lb.shape,
          "标签值", np.unique(np.asarray(lb.dataobj)))

## 第 6a 步：nnU-Net 实验规划（只规划，不预处理）

nnU-Net 自带的预处理会开多个进程，在 Colab 上内存不够会崩（报错 *Some background worker is 6 feet under*）。所以这里加 `--no_pp`，只做数据检查和实验规划；真正的预处理放到下一步，一个病人一个病人地做。

In [ ]:
!nnUNetv2_plan_and_preprocess -d 501 -c 3d_fullres --verify_dataset_integrity --no_pp -npfp 1
!df -h /content | tail -1

## 第 6b 步：逐个病人预处理（可断点续跑）

- 一次只处理一个病人，做完的会跳过，所以中途崩了重跑这一格就会接着做。
- 如果某个病人太大导致内核崩溃：重跑第 1、2 步，再跑这一格。程序会把上次崩溃的病人记进 `_skipped.txt` 并跳过它。
- 50 个病人大概要 1 个多小时。

In [ ]:
import os, json, gc, glob, shutil, psutil

assert "nnUNet_raw" in os.environ, "环境变量没了（内核重启过），请先重新运行第 1、2 步"

from nnunetv2.preprocessing.preprocessors.default_preprocessor import DefaultPreprocessor
from nnunetv2.utilities.plans_handling.plans_handler import PlansManager

DATASET_NAME = "Dataset501_HECKTOR"
raw_dir = f"{os.environ['nnUNet_raw']}/{DATASET_NAME}"
pp_dir  = f"{os.environ['nnUNet_preprocessed']}/{DATASET_NAME}"

plans_manager = PlansManager(f"{pp_dir}/nnUNetPlans.json")
cm = plans_manager.get_configuration("3d_fullres")
dataset_json = json.load(open(f"{pp_dir}/dataset.json"))
out_dir = f"{pp_dir}/{cm.data_identifier}"
os.makedirs(out_dir, exist_ok=True)
os.makedirs(f"{pp_dir}/gt_segmentations", exist_ok=True)
print("目标分辨率:", cm.spacing, " 中位图像大小:", cm.median_image_size_in_voxels)

def remove_outputs(p):
    # 删掉某个病人写了一半的预处理文件，否则训练时会读到坏文件
    for f in glob.glob(f"{out_dir}/{p}.*") + glob.glob(f"{out_dir}/{p}_seg.*") + \
             glob.glob(f"{pp_dir}/gt_segmentations/{p}.nii.gz"):
        os.remove(f)

# 如果上次在某个病人那里崩溃了，就把它记进跳过名单
marker = f"{pp_dir}/_working_on.txt"
skip_file = f"{pp_dir}/_skipped.txt"
if os.path.exists(marker):
    bad = open(marker).read().strip()
    print(f"⚠️ 上次处理 {bad} 时崩溃了，以后跳过它")
    remove_outputs(bad)
    with open(skip_file, "a") as f:
        f.write(bad + "\n")
    os.remove(marker)
skipped = set(open(skip_file).read().split()) if os.path.exists(skip_file) else set()

cases = sorted(f[:-7] for f in os.listdir(f"{raw_dir}/labelsTr")
               if f.endswith(".nii.gz") and f[:-7] not in skipped)

pre = DefaultPreprocessor(verbose=False)
for i, p in enumerate(cases, 1):
    if os.path.exists(f"{out_dir}/{p}.pkl"):
        continue  # 做过了，跳过（.pkl 是最后才写的，有它说明这个病人完整做完了）
    remove_outputs(p)
    open(marker, "w").write(p)
    mem = psutil.virtual_memory()
    print(f"[{i}/{len(cases)}] {p}   内存已用 {mem.used/1e9:.1f} / {mem.total/1e9:.1f} GB", flush=True)
    pre.run_case_save(f"{out_dir}/{p}",
                      [f"{raw_dir}/imagesTr/{p}_0000.nii.gz", f"{raw_dir}/imagesTr/{p}_0001.nii.gz"],
                      f"{raw_dir}/labelsTr/{p}.nii.gz",
                      plans_manager, cm, dataset_json)
    shutil.copy(f"{raw_dir}/labelsTr/{p}.nii.gz", f"{pp_dir}/gt_segmentations/{p}.nii.gz")
    os.remove(marker)
    gc.collect()

# 让 dataset.json 里的病例数和实际一致
for jpath in [f"{raw_dir}/dataset.json", f"{pp_dir}/dataset.json"]:
    d = json.load(open(jpath))
    d["numTraining"] = len(cases)
    json.dump(d, open(jpath, "w"), indent=4)

print("\n✅ 预处理完成！共", len(cases), "个病人", f"（跳过 {len(skipped)} 个）" if skipped else "")

## 第 7a 步：开始训练（第一次跑这一格）

- nnU-Net 默认要训 1000 个 epoch，T4 上每个 epoch 大约 6–7 分钟，太久了。
- 所以这里用 **50 个 epoch**（T4 大约 5–6 小时）。有 A100 可以改成 `nnUNetTrainer_100epochs`。
- 这里只训 5 折交叉验证里的第 0 折：40 个病人训练，10 个病人验证。
- **浏览器标签页别关**，不然 Colab 会断线。

In [ ]:
TRAINER = "nnUNetTrainer_50epochs"
!nnUNetv2_train 501 3d_fullres 0 -tr {TRAINER}

## 第 7b 步：断线后接着训练（只有断线时才跑这一格）

断线后：先重新跑第 1 到第 6 步，再跑这一格。`--c` 会从 Drive 上保存的进度继续。

In [ ]:
TRAINER = "nnUNetTrainer_50epochs"
!nnUNetv2_train 501 3d_fullres 0 -tr {TRAINER} --c

## 第 8 步：看结果（训练结束后跑）

In [ ]:
import json, os
from IPython.display import Image, display

TRAINER = "nnUNetTrainer_50epochs"
fold = f"{RESULTS}/{DATASET_NAME}/{TRAINER}__nnUNetPlans__3d_fullres/fold_0"

# 训练曲线：PPT 结果页用这张图
display(Image(f"{fold}/progress.png"))

# 验证集 Dice：PPT 结果页用这几个数字
s = json.load(open(f"{fold}/validation/summary.json"))
names = {"1": "GTVp 原发肿瘤", "2": "GTVn 淋巴结"}
for k, v in s["mean"].items():
    print(names.get(str(k), k), "Dice =", round(v["Dice"], 3))
print("平均 Dice =", round(s["foreground_mean"]["Dice"], 3))